# ProVe-Arabic — End-to-End Pipeline

Cross-lingual reference verification for Arabic Wikidata statements. Referenced by Appendix A.

Given a Wikidata statement (a triple: subject, property, object) and the page cited as its
reference, this pipeline decides whether that page supports the statement. The claim is in
Arabic. The reference may be in any language.

| Stage | Function |
|---|---|
| **A** Verbalisation | Triple → Arabic sentence (the *claim*) |
| **B** Retrieval + segmentation | Fetch reference, strip markup, split into candidate passages |
| **C/D** Entailment | Score each passage against the claim: supports / refutes / neutral |
| **E** Aggregation | Combine per-passage scores into one verdict |

**Requirements:** Colab with a **T4 GPU** (Runtime → Change runtime type → T4 GPU).
The trained model and all datasets download automatically on first run; no local setup or file placement is required.

**Run cell 1, then restart the runtime when Colab prompts**, then continue. The pinned install
replaces packages already loaded in memory, and Python does not reload imported modules.

**Verbaliser:** cell 4 loads the trained model by default. To rebuild it from the training
corpus instead, set `RETRAIN = True` in that cell (~15 min). Retraining writes to a separate
directory and never overwrites the published weights.

**Expected behaviour:** some references fail to fetch. Dead links, bot walls, and non-prose
pages are properties of the reference stock, not faults in the pipeline, quantified in
Section 5.3.2 of the dissertation. The pipeline reports these and continues.

**Companion notebooks:** `ProVe_Arabic_Evaluation.ipynb`,
`ProVe_Arabic_DatasetConstruction.ipynb`, `ProVe_Arabic_Benchmarks.ipynb`. Each is
self-contained and repeats the setup cells below.

## 1 — Environment  *(restart the runtime after this cell)*

In [ ]:
import os
# Pinned environment
!pip install -q --force-reinstall --no-deps "transformers==4.46.3"
!pip install -q pysbd camel-tools beautifulsoup4 lxml requests sentencepiece protobuf sacrebleu
!pip install -q "scikit-learn==1.8.0"

## 2 — Seeds, device, and data

Fixes all random seeds so results are reproducible, then downloads the datasets.

In [ ]:
import os, random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
GEN = torch.Generator(); GEN.manual_seed(SEED)      # passed to DataLoader so shuffling is fixed

# For bit-identical GPU results, uncomment the two lines below BEFORE any CUDA call
# They force deterministic kernels, but slow training and raise errors for ops that have no deterministic implementation
# Seeds alone give reproducible convergence
# os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
# torch.use_deterministic_algorithms(True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| seed:', SEED)

# Datasets
# Downloaded once from a public repository and cached locally in PD
from huggingface_hub import hf_hub_download
import shutil, os

DATA_REPO = 'ammarikhan003/prove-arabic-data'
PD = '/content/data'; os.makedirs(PD, exist_ok=True)

FILES = ['arabic_train_pairs_fixed.jsonl',                        # verbaliser training corpus (7,607 pairs)
         'val_fixed.json',                                         # held-out set for chrF
         'arabic_gold_candidates - arabic_gold_candidates.csv',     # annotated Arabic gold set (60 items)
         'wtr_claim_features.json',                                # cached claim-level feature vectors
         'gold_features.json',                                     # cached gold-set feature vectors
         'stage_e_rf.joblib']                                      # trained Stage E aggregator

for f in FILES:
    try:
        p = hf_hub_download(repo_id=DATA_REPO, filename=f, repo_type='dataset')
        shutil.copy(p, f'{PD}/{f}')
    except Exception as e:
        print('could not fetch', f, '->', type(e).__name__)
print('data ready in', PD)
for f in sorted(os.listdir(PD)): print('  ', f)

## 3 — Stage A: verbaliser  *(set `RETRAIN` here)*

In [ ]:
# TWO ROUTES, both defining the same verbalise() function:
#   RETRAIN = False  ->  download the trained model (seconds). DEFAULT
#   RETRAIN = True   ->  rebuild it from the training corpus (approx. 15 min on a T4)

# Retraining writes to a separate local directory, so it cannot overwrite the published weights
# Either route can be taken freely

RETRAIN = False

MODEL_REPO  = 'ammarikhan003/prove-arabic-mt5'
RETRAIN_DIR = '/content/mt5_retrained'

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

if not RETRAIN:
    TOK = AutoTokenizer.from_pretrained(MODEL_REPO)
    verbaliser = AutoModelForSeq2SeqLM.from_pretrained(MODEL_REPO).to(device).eval()
    print('verbaliser LOADED from', MODEL_REPO)

else:
    import json
    from torch.utils.data import DataLoader
    from transformers import get_linear_schedule_with_warmup

    TOK = AutoTokenizer.from_pretrained('google/mt5-small')
    verbaliser = AutoModelForSeq2SeqLM.from_pretrained('google/mt5-small').to(device)

    pairs = [json.loads(l) for l in open(f'{PD}/arabic_train_pairs_fixed.jsonl')]
    print('training pairs:', len(pairs))

    def collate(batch):
        """Tokenise a batch as a text-to-text task: delimited triple in, sentence out."""
        enc = TOK([x['input'] for x in batch], return_tensors='pt',
                  padding=True, truncation=True, max_length=64)
        lab = TOK(text_target=[x['target'] for x in batch], return_tensors='pt',
                  padding=True, truncation=True, max_length=96)['input_ids']
        # -100 is the value the loss function ignores
        # Without this the model is trained to predict padding tokens, which measurably degrades output quality
        lab[lab == TOK.pad_token_id] = -100
        enc['labels'] = lab
        return enc

    # generator=GEN makes the shuffle order deterministic, so a retrain is reproducible
    loader = DataLoader(pairs, batch_size=8, shuffle=True, collate_fn=collate, generator=GEN)
    opt = torch.optim.AdamW(verbaliser.parameters(), lr=3e-4)   # standard for T5 family
    steps = len(loader) * 3
    # 10% warmup then linear decay, as large early updates on a pretrained model are unstable
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)

    verbaliser.train()
    for ep in range(3):        # 3 epochs: loss plateaus here on this corpus size
        tot = 0.0
        for i, b in enumerate(loader):
            b = {k: v.to(device) for k, v in b.items()}
            loss = verbaliser(**b).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(verbaliser.parameters(), 1.0)   # prevents loss spikes
            opt.step(); sched.step(); opt.zero_grad()
            tot += loss.item()
            if i % 200 == 0:
                print(f'  ep{ep+1} step {i}/{len(loader)} loss {loss.item():.3f}')
        print(f'epoch {ep+1}/3 avg loss {tot/len(loader):.3f}')
    verbaliser.eval()
    verbaliser.save_pretrained(RETRAIN_DIR); TOK.save_pretrained(RETRAIN_DIR)
    print('verbaliser RETRAINED and saved to', RETRAIN_DIR)


def verbalise(subj, prop, obj):
    """Turn a triple into an Arabic sentence.

    Decoding choices, all addressing observed failure modes:
      num_beams=4            beam search considers several continuations before
                             committing, giving more fluent output than greedy decoding
      no_repeat_ngram_size=3 forbids any 3-token sequence from recurring, which fixes a
                             degenerate mode where a phrase repeated within one sentence
      repetition_penalty=1.2 a softer discouragement of reusing tokens already generated
    All three act at decoding time, so they required no retraining.
    """
    enc = TOK(f'{subj} | {prop} | {obj}', return_tensors='pt',
              truncation=True, max_length=64).to(device)
    with torch.no_grad():
        o = verbaliser.generate(**enc, max_length=96, num_beams=4,
                                no_repeat_ngram_size=3, repetition_penalty=1.2)
    return TOK.decode(o[0], skip_special_tokens=True)


# Smoke test on a property that the inversion fix was written to repair
print('smoke test:', verbalise('دوغلاس آدمز', 'مكان الولادة', 'كامبريدج'))


# 4 — Stage B: retrieval and Arabic-aware segmentation

- **`fetch_html`** — browser-like user-agent (many sites reject the default Python agent, which
  would inflate the apparent dead-link rate) and a timeout (some hosts accept a connection then
  never respond).
- **`clean_to_text`** — strips scripts, styles, navigation, headers, footers. Site boilerplate
  repeats on every page and is never claim-specific; keeping it would add hundreds of irrelevant
  passages per document.
- **`detect_lang`** — routes on the proportion of Arabic-script characters. The decision is
  binary, so a character count suffices and a language-ID model would be overkill.
- **`normalize_ar`** — unifies Arabic orthographic variants (chiefly the forms of *alif*) that
  would otherwise make one word appear as several distinct strings.
- **`segment`** — pysbd splitting, then a merge that rejoins fragments until a real sentence
  terminator. The terminator set includes Arabic punctuation (؟ ؛), distinct codepoints from
  their Latin counterparts.

In [ ]:
!pip install -q pysbd camel-tools beautifulsoup4 lxml requests

import re, requests, pysbd
from urllib.parse import urlsplit, urlunsplit, quote
from bs4 import BeautifulSoup

# Arabic has several written forms of the same letter (notably alif: ا إ أ آ),
# so the same word can appear as different strings.
# Normalising collapses them, which matters because the entailment model treats distinct strings as distinct tokens.
try:
    from camel_tools.utils.normalize import (normalize_alef_ar,
                                             normalize_alef_maksura_ar,
                                             normalize_teh_marbuta_ar)
    from camel_tools.utils.dediac import dediac_ar
    def normalize_ar(t):
        # dediac strips short-vowel diacritics, the three normalisers unify letter variants
        return normalize_teh_marbuta_ar(normalize_alef_maksura_ar(normalize_alef_ar(dediac_ar(t))))
except Exception:
    # Regex fallback so the pipeline degrades rather than fails if CAMeL Tools is unavailable
    def normalize_ar(t):
        t = re.sub(r'[\u064B-\u0652\u0670\u0640]', '', t)   # remove diacritics and tatweel
        t = re.sub(r'[إأآا]', 'ا', t)                          # unify alif forms
        return t.replace('ى', 'ي').replace('ة', 'ه')           # unify yaa / teh marbuta

# The reference may be in any language
# This is a cross-lingual pipeline: an Arabic claim is checked against evidence in whatever language the source happens to be.
# Only a binary decision is needed so counting characters in the Arabic Unicode blocks is sufficient and avoids a heavier language-ID model.
AR = re.compile(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]')

def arabic_ratio(t):
    letters = [c for c in t if c.isalpha()]
    return sum(bool(AR.match(c)) for c in letters) / len(letters) if letters else 0.0

def detect_lang(t):
    # 0.4 rather than 0.5
    # Arabic pages routinely contain inline Latin-script technical terms, proper nouns, and URLs
    # which would otherwise flip the classification.
    return 'ar' if arabic_ratio(t) >= 0.4 else 'en'

# A conventional User-Agent is necessary
# Many sites reject the default agent sent by Python HTTP clients
# which would show up as a dead reference and inflate the apparent failure rate of the reference stock.
HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/124.0 Safari/537.36 ProVe-Arabic-research/1.0'),
    'Accept-Language': 'ar,en;q=0.9',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

def _safe(u):
    # Percent-encode the path so non-ASCII URLs (e.g. Arabic Wikipedia titles) are valid
    p = urlsplit(u)
    return urlunsplit((p.scheme, p.netloc, quote(p.path), p.query, p.fragment))

def fetch_html(url):
    # timeout guards against hosts that accept a connection and never respond
    # which would otherwise stall a batch run indefinitely
    r = requests.get(_safe(url), headers=HEADERS, timeout=25)
    r.raise_for_status()
    r.encoding = r.apparent_encoding or r.encoding   # guess encoding, wrong guesses ruin Arabic
    return r.text

# Elements that never contain claim-specific prose.
JUNK_TAGS = ['script','style','noscript','nav','footer','header','aside',
             'form','button','svg','iframe','figure','sup']
# Block-level elements that do carry prose.
# Extracting per-block preserves the document's natural boundaries, which helps the segmenter.
BLOCKS = ['p','li','h1','h2','h3','h4','h5','h6','td','th','blockquote','caption','dd','dt']

LISTTOGGLE = re.compile(r'^\s*القائمة\s*\.{2,}')   # Arabic Wikipedia "menu ...." UI artefact
PUNCT_FIX = [(re.compile(r'\s+([.,؛،:!؟…\)\]])'), r'\1'),   # no space before closing punctuation
             (re.compile(r'([(\[])\s+'), r'\1'),             # no space after opening bracket
             (re.compile(r'\s{2,}'), ' ')]                    # collapse runs of whitespace

def tidy_spacing(t):
    for p, r in PUNCT_FIX:
        t = p.sub(r, t)
    return t.strip()

def clean_to_text(html):
    soup = BeautifulSoup(html, 'lxml')
    for t in soup(JUNK_TAGS):
        t.decompose()                       # remove boilerplate entirely
    seen, chunks = set(), []
    for b in soup.find_all(BLOCKS):
        txt = tidy_spacing(LISTTOGGLE.sub('', b.get_text(' ', strip=True)))
        if not txt or txt in seen:
            continue                        # skip empties and exact duplicates
        seen.add(txt)
        if txt[-1] not in '.!?؟…؛،:':
            txt += '.'                      # give the segmenter a terminator to split on
        chunks.append(txt)
    text = '\n'.join(chunks)
    if len(text) < 200:
        # Block extraction found almost nothing
        # Fall back to the whole document
        text = tidy_spacing(soup.get_text('\n', strip=True))
    return re.sub(r'\n{2,}', '\n', text).strip()

# Arabic sentence terminators
TERMINALS = '.؟?!…؛'
_seg = {}

def _segmenter(lang):
    # cached, constructing a pysbd Segmenter is not free, and this runs per document
    if lang not in _seg:
        _seg[lang] = pysbd.Segmenter(language=('ar' if lang == 'ar' else 'en'), clean=False)
        # clean=False: pysbd's own cleaning is tuned for English typography and
        # interacts badly with text already cleaned above
    return _seg[lang]

def segment(text, lang, min_chars=20):
    """Split cleaned text into candidate evidence passages (one sentence each)."""
    seg = _segmenter(lang)
    raw = []
    for line in text.split('\n'):          # segment block by block, not whole-document
        line = line.strip()
        if line:
            raw += [s.strip() for s in seg.segment(line) if s.strip()]

    # terminal merge
    merged, buf = [], ''
    for s in raw:
        buf = f'{buf} {s}'.strip() if buf else s
        if buf[-1] in TERMINALS:
            merged.append(buf); buf = ''
    if buf:
        merged.append(buf)                  # flush any trailing unterminated text

    # filtering
    seen, out = set(), []
    for s in merged:
        if s in seen:
            continue                        # a repeated caption would otherwise be counted twice
        if sum(c.isalpha() for c in s) < 5 or len(s) < min_chars:
            continue                        # very short fragments attract spuriously
                                            # confident stance scores from the NLI model
        seen.add(s); out.append(s)
    return out

print('Stage B ready: fetch_html, clean_to_text, detect_lang, normalize_ar, segment')


## 5 — Stage C/D: cross-lingual entailment

**Argument order matters:** the *passage* is the premise, the *claim* is the hypothesis. "Given
what this passage says, what follows about the claim?" Reversed, it asks whether the claim
implies the passage, a different and incorrect question.

**Label safeguard:** the index→class mapping is read from the model config. NLI
checkpoints do not share a standard ordering, and a wrong assumption raises no error. It
silently swaps two classes, producing results that look plausible and are wrong.

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# mDeBERTa-v3 fine-tuned on MNLI + XNLI
NLI = 'MoritzLaurer/mDeBERTa-v3-base-mnli-xnli'
nli_tok = AutoTokenizer.from_pretrained(NLI)
nli = AutoModelForSequenceClassification.from_pretrained(NLI).to(device).eval()

# read the index-to-class mapping from the model's own config
id2label = {int(k): v.lower() for k, v in nli.config.id2label.items()}
print('device:', device, '| label map:', id2label)


def entail_scores(premise, hypothesis):
    """Score one passage against one claim.

    premise    = the evidence passage from the reference document
    hypothesis = the claim being verified

    Argument order is load-bearing: this asks "given what the passage says, what
    follows about the claim?" Reversing them asks whether the claim implies the
    passage, which is a different and incorrect question.

    Returns probabilities for {entailment, neutral, contradiction}, which map onto
    the verification labels SUPPORTS / NEI / REFUTES respectively.
    """
    x = nli_tok(premise, hypothesis, return_tensors='pt',
                truncation=True, max_length=256).to(device)   # cap bounds memory use
    with torch.no_grad():                                     # no gradients at inference, approx. half the memory
        probs = F.softmax(nli(**x).logits[0], dim=-1)         # logits -> calibrated probabilities
    return {id2label[i]: probs[i].item() for i in range(len(probs))}


# An ARABIC claim scored against ENGLISH passages
# One supporting, one contradicting, one merely related.
claim = "وُلد دوغلاس آدمز في كامبريدج"          # "Douglas Adams was born in Cambridge"
demo = [
    "Douglas Adams was born in Cambridge in 1952.",                # should be entailment
    "Douglas Adams was a British author born in London.",          # should be contradiction
    "Douglas Adams wrote The Hitchhiker's Guide to the Galaxy.",   # should be neutral
]
print(f'\nCLAIM (Arabic): {claim}\n')
for p in demo:
    s = entail_scores(p, claim)
    print(f"  entail {s['entailment']:.2f} | contradict {s['contradiction']:.2f} "
          f"| neutral {s['neutral']:.2f}  <-  {p}")


## 6 — Stage E: stance aggregation

Reduces many conflicting per-passage stances to one verdict. Three strategies are available.

**`learned`** (default) is the Random Forest adopted in the dissertation, trained on the WTR
benchmark in `ProVe_Arabic_Evaluation.ipynb` and downloaded here as a fitted model. It reads a
thirteen-dimensional summary of the document's score distribution.

**`rule`** (supports if any passage supports, else refutes if any refutes, else NEI) and
**`weighted`** (class probabilities summed, weighted by engagement = 1 − neutral, used as a
proxy because collapsing Stages C and D removed ProVe's separate relevance score) are the
training-free alternatives, retained so the three can be compared on the same evidence.

Section 5.6.2 reports the learned aggregator at 0.6800 accuracy against the rule's 0.4820.

In [ ]:
# Three strategies. `learned` is the Random Forest adopted
# The two training-free strategies from the fact-checking literature are retained for comparison
#
# The forest was trained on WTR's archived English references
# So applying it to live pages here is a cross-benchmark transfer, the same setting as the gold-set evaluation in §5.7.2.

import joblib, numpy as np

RF = joblib.load(f'{PD}/stage_e_rf.joblib')
SUPPORTS_IDX = list(RF.classes_).index('SUPPORTS')   # column of predict_proba to read


def featurise(scored):
    """Reduce a variable number of passages to the fixed 13-feature vector the forest expects.

    Feature order MUST match the training code in ProVe_Arabic_Evaluation.ipynb — a mismatch
    raises no error, the forest simply reads the wrong quantities.
    """
    e = np.array([r['entail']     for r in scored])
    c = np.array([r['contradict'] for r in scored])
    n = np.array([r['neutral']    for r in scored])
    es, cs = np.sort(e)[::-1], np.sort(c)[::-1]
    top = lambda a, k: float(a[k]) if len(a) > k else 0.0
    return [[float(e.max()), float(e.mean()), top(es, 1), top(es, 2),   # support evidence
             float(c.max()), float(c.mean()), top(cs, 1), top(cs, 2),   # refute evidence
             float(n.mean()),                                          # how neutral overall
             float((e > 0.5).sum()), float((c > 0.5).sum()),            # counts of strong passages
             float(len(scored)),                                        # page length
             float(e.max() - c.max())]]                                 # support vs refute margin


def aggregate(scored, strategy='learned'):
    """Combine per-passage scores into (verdict, support_probability).

    scored = list of dicts, each with keys entail / contradict / neutral for one passage.
    """
    if strategy == 'learned':
        X = featurise(scored)
        return RF.predict(X)[0], float(RF.predict_proba(X)[0][SUPPORTS_IDX])

    if strategy == 'rule': # Rule-based
        preds = [max(('SUPPORTS', r['entail']),
                     ('REFUTES',  r['contradict']),
                     ('NEI',      r['neutral']),
                     key=lambda kv: kv[1])[0] for r in scored]
        if   'SUPPORTS' in preds: z = 'SUPPORTS'
        elif 'REFUTES'  in preds: z = 'REFUTES'
        else:                     z = 'NEI'
        return z, (1.0 if z == 'SUPPORTS' else 0.0)

    # Weighted sum
    mu = {'SUPPORTS': 0.0, 'REFUTES': 0.0, 'NEI': 0.0}
    for r in scored:
        w = max(0.0, 1.0 - r['neutral'])
        mu['SUPPORTS'] += w * r['entail']
        mu['REFUTES']  += w * r['contradict']
        mu['NEI']      += w * r['neutral']
    z = max(mu, key=mu.get)
    tot = sum(mu.values()) or 1.0          # guard against division by zero
    return z, mu['SUPPORTS'] / tot          # graded support probability


def verify(claim, passages, topk=5, strategy='learned'):
    """Full Stage C/D + E: score every passage, aggregate, return the most engaged for display."""
    scored = []
    for p in passages:
        s = entail_scores(p, claim)
        scored.append({'entail':     s.get('entailment', 0),
                       'contradict': s.get('contradiction', 0),
                       'neutral':    s.get('neutral', 0),
                       'passage':    p})

    # Aggregate over ALL passages: n_passages and the mean features describe the whole
    # document, so truncating first would change what the forest sees.
    verdict, support_prob = aggregate(scored, strategy)

    # Rank by how strongly the model committed either way.
    # Passages it found neutral carry no information about the claim, so they are the ones to drop.
    scored.sort(key=lambda r: max(r['entail'], r['contradict']), reverse=True)
    return verdict, support_prob, scored[:topk]


# All three strategies on the same evidence, to show they can disagree
claim = "وُلد دوغلاس آدمز في كامبريدج"
passages = ["Douglas Adams was born in Cambridge in 1952.",
            "Douglas Adams was a British author born in London.",
            "Douglas Adams wrote The Hitchhiker's Guide to the Galaxy."]
for strat in ('learned', 'weighted', 'rule'):
    v, y, top = verify(claim, passages, strategy=strat)
    print(f'[{strat:8s}] verdict={v}  support_prob={y:.4f}')


## 7 — Sample real claim–reference pairs from Wikidata

Queries the Wikidata SPARQL endpoint for statements carrying a reference URL, seeded on fixed
entities so the sample is reproducible. Keeps six with distinct references and runs Stage B on
each, so retrieval can be inspected before the reasoning stages are attached.

In [ ]:
# Pulls genuine Wikidata statements that carry a reference URL
# Seeded on four fixed entities (Douglas Adams, Isaac Newton, Naguib Mahfouz, Albert Einstein)
# The sample is the same on every run and results are comparable

import requests

# 1. query Wikidata for referenced statements
# prov:wasDerivedFrom/pr:P854 is the path from a statement to its reference URL
# Only statements that actually cite a source are returned
# Property labels are requested in Arabic, since the claim must be verbalised in Arabic
SPARQL = """
SELECT ?item ?itemLabel ?propLabel ?value ?valueLabel ?ref WHERE {
  VALUES ?item { wd:Q42 wd:Q937 wd:Q7251 wd:Q935 }
  ?item ?p ?st .
  ?st prov:wasDerivedFrom/pr:P854 ?ref .
  ?prop wikibase:claim ?p ; wikibase:statementProperty ?ps .
  ?st ?ps ?value .
  ?prop rdfs:label ?propLabel . FILTER(LANG(?propLabel) = "ar")
  SERVICE wikibase:label { bd:serviceParam wikibase:language "ar,en". }
}
LIMIT 50
"""
r = requests.get("https://query.wikidata.org/sparql",
                 params={"query": SPARQL, "format": "json"},
                 headers={"User-Agent": "ProVe-Arabic-research/1.0 (KCL MSc dissertation)"},
                 timeout=60)
rows = r.json()["results"]["bindings"]
print(f"{len(rows)} referenced statements returned\n")

claims = []
for b in rows:
    claims.append({
        "subj": b["itemLabel"]["value"],
        "prop": b["propLabel"]["value"],
        "obj":  b.get("valueLabel", b.get("value"))["value"],
        "ref":  b["ref"]["value"],
    })

# Keep six pairs with DISTINCT reference URLs
seen, sample = set(), []
for c in claims:
    if c["ref"] in seen: continue
    seen.add(c["ref"]); sample.append(c)
    if len(sample) >= 6: break

# 2. Run Stage B on each reference
# Inspecting retrieval on its own, before the reasoning stages are attached
# A later wrong verdict can be attributed to the right stage
for c in sample:
    triple = f'{c["subj"]} | {c["prop"]} | {c["obj"]}'
    print("TRIPLE :", triple)
    print("REF    :", c["ref"])
    try:
        t = clean_to_text(fetch_html(c["ref"]))
        lang = detect_lang(t)
        sents = segment(t, lang)
        print(f"         [{lang}, {len(sents)} candidate passages]")
        for s in sents[:3]:
            print("   •", s[:160])
    except Exception as e:
        print("         (fetch/segment failed:", type(e).__name__, "- dead link, PDF, or JS page)")
    print()

## 8 — Full pipeline: A → B → C/D → E

The complete chain on the sampled pairs: verbalise the triple, fetch and segment the reference,
score every passage, aggregate to a verdict. References yielding fewer than three passages are
skipped as likely blocked or empty, matching the unprocessable-reference policy of Section 5.1.3.

In [ ]:
# Note what the output demonstrates
# The claim is Arabic while most references are English, so every verdict below is a cross-lingual judgement.

for c in sample:
    claim = verbalise(c['subj'], c['prop'], c['obj'])        # STAGE A
    print("=" * 70)
    print("TRIPLE:", f'{c["subj"]} | {c["prop"]} | {c["obj"]}')
    print("CLAIM :", claim)
    print("REF   :", c['ref'])

    # STAGE B
    # Failures here are expected and are a property of the reference stock.
    try:
        t = clean_to_text(fetch_html(c['ref']))
        lang = detect_lang(t)
        passages = segment(t, lang)
    except Exception as e:
        print("        (reference unavailable:", type(e).__name__, ")")
        continue

    # Evaluation policy in Section 5.1.3.
    if len(passages) < 3:
        print(f"        [{lang}, {len(passages)} passages - likely blocked or empty, skipping]")
        continue

    # STAGES C/D + E
    verdict, support_prob, top = verify(claim, passages, strategy='learned')
    print(f"        [{lang}, {len(passages)} passages]  ->  VERDICT: {verdict}  (support {support_prob:.4f})")

    # Show the three most-engaged passages
    for r in top[:3]:
        print(f"        e{r['entail']:.2f} c{r['contradict']:.2f} n{r['neutral']:.2f}  {r['passage'][:140]}")
